In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

# ========= PATHS =========
# Your multi-sheet Excel file in Drive
input_path  = "/content/drive/MyDrive/Major Project/Dataset.xlsx"

# Name of the sheet you want (2nd sheet: AQI_station_day)
sheet_name = "AQI_station_day"   # or sheet_name=1 if you prefer index

# Output cleaned file
output_path = "/content/drive/MyDrive/Major Project/AQI_station_day_cleaned.xlsx"
# =========================

# 1) Read only the desired sheet
df = pd.read_excel(input_path, sheet_name=sheet_name)

print("Columns in file:")
print(df.columns.tolist())

# 2) Columns to check for all-zero/empty
cols_to_check = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene',
    'Xylene', 'AQI', 'AQI_Bucket'
]

missing = [c for c in cols_to_check if c not in df.columns]
if missing:
    raise ValueError(f"These columns are missing from this sheet: {missing}")

# 3) Treat 0 / '0' / '0.0' / blank / NaN as zero-or-null
def is_zero_or_null(x):
    if pd.isna(x):
        return True
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return True
        try:
            return float(x) == 0
        except ValueError:
            return False
    try:
        return float(x) == 0
    except (TypeError, ValueError):
        return False

# 4) True where ALL pollutant columns are zero-or-null
mask_all_zero_null = df[cols_to_check].applymap(is_zero_or_null).all(axis=1)

# 5) Keep only rows that are NOT all-zero/null in those columns
df_clean = df[~mask_all_zero_null].copy()

print(f"Original rows: {len(df)}")
print(f"Rows removed (all pollutant cols = 0/empty): {mask_all_zero_null.sum()}")
print(f"Rows after cleaning: {len(df_clean)}")

# 6) Sanity-check that station_name, city, state are intact
for col in ["station_name", "city", "state"]:
    if col in df_clean.columns:
        print(f"\nSample values from '{col}':")
        print(df_clean[col].head())
    else:
        print(f"\nWARNING: column '{col}' not found in dataframe")

# 7) Save cleaned sheet
df_clean.to_excel(output_path, index=False)

print("\nCleaned file saved to:")
print(output_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Columns in file:
['StationId', 'Station name', 'city', 'State', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']


/tmp/ipython-input-2035708434.py:53: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask_all_zero_null = df[cols_to_check].applymap(is_zero_or_null).all(axis=1)


Original rows: 105364
Rows removed (all pollutant cols = 0/empty): 9645
Rows after cleaning: 95719


Sample values from 'city':
0    Amaravati
1    Amaravati
2    Amaravati
3    Amaravati
4    Amaravati
Name: city, dtype: object


Cleaned file saved to:
/content/drive/MyDrive/Major Project/AQI_station_day_cleaned.xlsx
